In [ ]:
import pandas as pd 
dataset = "v3"
df_treino = pd.read_csv(f"../datasets/df_treino_{dataset}.csv")
[col for col in df_treino.columns if not str(col).isidentifier()]

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

dataset = "v3"
df_treino = pd.read_csv(f"../datasets/df_treino_{dataset}.csv")
df_treino = df_treino.drop(columns= ["cavidade"])

y=df_treino["target"].copy()
X=df_treino.drop(columns=["target"]).copy()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,

    test_size=0.2,
    random_state=42,
    stratify=y
)

df_treino.shape

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(
    X_train,
    y_train
)

y_prob = xgb.predict_proba(X_test)[:,1]


for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    print(f"\nXGBOOST - Dataset {dataset}:\nThreshold: {threshold}")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
tree = DecisionTreeClassifier(
    max_depth=20,
    random_state=42,
    class_weight="balanced"
)

tree.fit(
    X_train,
    y_train
)

y_prob = tree.predict_proba(X_test)[:,1]


for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    print(f"\nDecision Tree - Dataset {dataset}\nThreshold: {threshold}")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )


In [ ]:
from sasctl.pzmm import PickleModel
from sasctl.pzmm import ScoreCode
from sasctl.pzmm import JSONFiles
from sasctl.pzmm import ZipModel

help(JSONFiles)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
from sasctl.pzmm import PickleModel
from sasctl.pzmm import ScoreCode
from sasctl.pzmm import JSONFiles
from sasctl.pzmm import ZipModel
from pathlib import Path

random_forest = RandomForestClassifier(
    n_estimators=1400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

y_prob = random_forest.predict_proba(X_test)[:,1]


for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    print(f"\nRandom forest - Dataset {dataset}\nThreshold: {threshold}")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )
joblib.dump(random_forest, f"../models/random_forest_{dataset}.pkl")
PickleModel.pickle_trained_model(
    f"Espeleo-ML-RF-{dataset}",
    random_forest,
    "../sas_model"
)
sc = ScoreCode()
sc.write_score_code(
    model_prefix= f"Espeleo-ML-RF-{dataset}",
    input_data=X_train,
    target_variable="target",
    predict_method=[random_forest.predict_proba,
                    [float, float]],
    target_index=1,
    target_values=[0,1],
    score_code_path="../sas_model",
    model_file_name=f"Espeleo-ML-RF-{dataset}.pickle",
    predict_threshold=0.4
)




model_prefix = f"Espeleo-ML-RF-{dataset}"

model_path = Path("../sas_model") / model_prefix
model_path.mkdir(parents=True, exist_ok=True)

JSONFiles.write_var_json(
    input_data=X_train,
    is_input=True,
    json_path=model_path
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
logistic_reg = LogisticRegression(
    max_iter=4000,
    class_weight="balanced"
)

logistic_reg.fit(
    X_train,
    y_train
)

y_pred = logistic_reg.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
import pandas as pd
df_treino = pd.read_csv("../datasets/df_treino_v3.csv")
df_treino = df_treino.drop(columns=["cavidade"])

columns = df_treino.columns.to_list()

fields = {}
TYPE_MAP = {
    "object": "str",
    "int64": "int",
    "float64": "float",
}
for column, type in df_treino.dtypes.items():
    print(f"    {column} : {TYPE_MAP[str(type)]}")
 

In [ ]:
import pandas as pd
import json
df_treino = pd.read_csv("../datasets/df_treino_v3.csv")
df_treino = df_treino.drop(columns=["cavidade"])

df_treino = df_treino[
    df_treino["target"] ==1
]
columns = df_treino.columns.to_list()

fields = {}

sample = df_treino.sample(1).iloc[0].to_dict()



print(json.dumps(
    sample,
    indent=4
))